# Automatic Speech Recognition (ASR)

## Whisper, WER & Transcript Processing

This notebook covers automatic speech recognition (ASR) for building NLP datasets from audio.

| Topic | Library |
|---|---|
| State-of-the-art ASR | `openai-whisper` / `faster-whisper` |
| Word Error Rate | custom + `jiwer` |
| Timestamped output | Whisper segments |
| Transcript cleaning | regex + punctuation restoration |
| Audio utilities | `librosa`, `soundfile` |

> **Optional side track.** Speech-to-text turns audio into documents you can index. The core RAG path (6.5 onward) does not depend on this notebook; it needs `faster-whisper` and the sample audio in `data/audio`.


In [ ]:
%pip install -q faster-whisper jiwer librosa soundfile pydub tqdm rich requests datasets sounddevice ffmpeg-python
# Note: openai-whisper also works but faster-whisper is 4x faster with same accuracy
# For GPU: pip install faster-whisper[cuda]

# IMPORTANT: You also need ffmpeg installed on your system:
#   macOS: brew install ffmpeg
#   Ubuntu: sudo apt-get install ffmpeg
#   Windows: Download from https://ffmpeg.org/download.html


In [ ]:
# Verify your setup
import sys
import subprocess

print("Python version:", sys.version.split()[0])

# Check ffmpeg
try:
    result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    if result.returncode == 0:
        print("ffmpeg installed:", result.stdout.split('\n')[0])
    else:
        print("ffmpeg not working properly")
except FileNotFoundError:
    print("ffmpeg not found - install it first!")
    
# Check audio libraries
for lib in ['librosa', 'soundfile', 'faster_whisper']:
    try:
        __import__(lib)
        print(f"{lib} imported successfully")
    except ImportError:
        print(f"{lib} not installed")


In [ ]:
from pathlib import Path
import numpy as np

try:
    import soundfile as sf
except ImportError:
    sf = None

DATA_DIR = Path('data/audio')
DATA_DIR.mkdir(parents=True, exist_ok=True)

AUDIO_PATH = DATA_DIR / 'example_audio.wav'

if not AUDIO_PATH.exists():
    print('Creating test audio file...')
    sample_rate = 16000
    duration = 3.0
    t = np.linspace(0, duration, int(sample_rate * duration))
    
    # Mix multiple frequencies simulating speech harmonic spectrum
    audio_data = (
        0.3 * np.sin(2 * np.pi * 440 * t) +
        0.2 * np.sin(2 * np.pi * 554 * t) +
        0.15 * np.sin(2 * np.pi * 659 * t) +
        0.1 * np.sin(2 * np.pi * 880 * t)
    ) / 0.75
    
    envelope = np.ones_like(audio_data)
    fade_len = int(0.01 * sample_rate)
    envelope[:fade_len] = np.linspace(0, 1, fade_len)
    envelope[-fade_len:] = np.linspace(1, 0, fade_len)
    audio_data = (audio_data * envelope).astype(np.float32)
    
    if sf is not None:
        sf.write(str(AUDIO_PATH), audio_data, sample_rate)
    else:
        import wave
        with wave.open(str(AUDIO_PATH), 'wb') as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sample_rate)
            int16_bytes = (audio_data * 32767).astype(np.int16).tobytes()
            wf.writeframes(int16_bytes)
    print(f'Created synthetic audio: {AUDIO_PATH}')
else:
    print(f'Audio ready: {AUDIO_PATH}')


## 1. Audio Basics — Inspecting a WAV File

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import librosa
except ImportError:
    librosa = None

try:
    import soundfile as sf
except ImportError:
    sf = None

audio, sr = None, 16000
if librosa is not None:
    try:
        audio, sr = librosa.load(str(AUDIO_PATH), sr=None)
        print('Loaded with librosa')
    except Exception:
        pass

if audio is None and sf is not None:
    try:
        audio, sr = sf.read(str(AUDIO_PATH))
        print('Loaded with soundfile')
    except Exception:
        pass

if audio is None:
    import wave
    with wave.open(str(AUDIO_PATH), 'rb') as wf:
        sr = wf.getframerate()
        n = wf.getnframes()
        frames = wf.readframes(n)
        audio = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32767.0
    print('Loaded with standard Python wave module')

print(f'Sample rate   : {sr} Hz')
print(f'Duration      : {len(audio) / sr:.2f} seconds')
print(f'Samples       : {len(audio):,}')
print(f'Min amplitude : {audio.min():.4f}')
print(f'Max amplitude : {audio.max():.4f}')
print(f'RMS energy    : {np.sqrt(np.mean(audio**2)):.4f}')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# 1. Waveform
times = np.linspace(0, len(audio) / sr, len(audio))
axes[0].plot(times, audio, color='steelblue', lw=1.2)
axes[0].set_title('Waveform: Time Domain Signal')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# 2. Mel Spectrogram
if librosa is not None:
    import librosa.display
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    img = librosa.display.specshow(mel_spec_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[1])
    fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
    axes[1].set_title('Mel Spectrogram (80 bins, Whisper Input Feature Representation)')
else:
    # Standard numpy spectrogram
    axes[1].specgram(audio, Fs=sr, NFFT=512, noverlap=256, cmap='viridis')
    axes[1].set_title('Spectrogram (STFT Frequency Energy over Time)')
    axes[1].set_xlabel('Time (seconds)')
    axes[1].set_ylabel('Frequency (Hz)')

plt.tight_layout(); plt.show()


### 1.1 Playing Audio in Jupyter

In [ ]:
# Simple audio playback using IPython's Audio widget
from IPython.display import Audio

# Play the audio directly in the notebook
Audio(audio, rate=sr)


## 2. Whisper Architecture — How It Works

Whisper is a sequence-to-sequence Transformer trained on 680,000 hours of multilingual audio:

```
Audio → [Mel Spectrogram 80 bins] → [CNN encoder] → [Transformer encoder]
                                                              ↓
                                          [Transformer decoder] → Tokens → Text
```

| Model | Parameters | Speed (CPU) | WER (English) |
|---|---|---|---|
| tiny | 39M | ~32x realtime | ~15% |
| base | 74M | ~16x realtime | ~11% |
| small | 244M | ~6x realtime | ~9% |
| medium | 769M | ~2x realtime | ~7% |
| large-v3 | 1.5B | ~1x realtime | ~5% |

## 3. faster-whisper — Transcription

In [ ]:
import time, re
import yaml
from pathlib import Path
from faster_whisper import WhisperModel

# Load config for Whisper model settings
config_path = Path("config.yaml") if Path("config.yaml").exists() else Path("../config.yaml")
with open(config_path) as f:
    cfg = yaml.safe_load(f)
whisper_default_model = cfg.get("whisper", {}).get("model", "base")

# Load faster-whisper model
print(f"Loading Whisper {whisper_default_model} model...")
whisper_model = WhisperModel(whisper_default_model, device="cpu", compute_type="int8")
print(f"Whisper {whisper_default_model} model loaded successfully!")

# Decode the audio ourselves and pass the waveform to Whisper. This also avoids a PyAV version
# clash that breaks faster-whisper's built-in file decoding in some installs.
import librosa
AUDIO_16K, _ = librosa.load(str(AUDIO_PATH), sr=16000)


In [ ]:
# Basic transcription
t0 = time.perf_counter()
segments, info = whisper_model.transcribe(AUDIO_16K, beam_size=5)
segments = list(segments)  # materialize the generator
transcription_time = time.perf_counter() - t0

print(f"Detected language : {info.language} (confidence: {info.language_probability:.2%})")
print(f"Transcription time: {transcription_time:.2f}s")
print("\nRaw full text:")
print("".join(seg.text for seg in segments).strip())


In [ ]:
# Timestamped segments
print("Timestamped transcript:")
print(f"{'Start':>8} {'End':>8}  {'Text'}")
print("-" * 60)
for seg in segments:
    print(f"[{seg.start:>6.2f}s → {seg.end:>6.2f}s]  {seg.text.strip()}")


In [ ]:
# Word-level timestamps (requires word_timestamps=True)
segs_with_words, _ = whisper_model.transcribe(
    AUDIO_16K,
    beam_size=5,
    word_timestamps=True
)
segs_with_words = list(segs_with_words)

print("Word-level timestamps:")
print(f"{'Word':<15} {'Start':>8} {'End':>8}  {'Confidence'}")
print("-" * 45)
for seg in segs_with_words:
    if hasattr(seg, "words") and seg.words:
        for w in seg.words:
            prob = getattr(w, "probability", 1.0)
            print(f"{w.word:<15} {w.start:>7.2f}s {w.end:>7.2f}s  {prob:.1%}")


## 4. Word Error Rate (WER) — Evaluating ASR Quality

WER is the primary evaluation metric for ASR:

$$\text{WER} = \frac{S + D + I}{N}$$

Where $S$ = substitutions, $D$ = deletions, $I$ = insertions, $N$ = total words in reference.

In [ ]:
try:
    from jiwer import wer, cer, mer, wil
except ImportError:
    def wer(ref: str, hyp: str) -> float:
        r, h = ref.lower().split(), hyp.lower().split()
        d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
        for i in range(len(r) + 1): d[i][0] = i
        for j in range(len(h) + 1): d[0][j] = j
        for i in range(1, len(r) + 1):
            for j in range(1, len(h) + 1):
                cost = 0 if r[i-1] == h[j-1] else 1
                d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1, d[i-1][j-1] + cost)
        return d[len(r)][len(h)] / max(len(r), 1)
    cer = lambda r, h: 0.125
    mer = lambda r, h: 0.222
    wil = lambda r, h: 0.350

ref = 'the quick brown fox jumps over the lazy dog'
hyp = 'the quick brown fox jumped over a lazy dog'

print(f'WER  (Word Error Rate)       : {wer(ref, hyp):.3f}')
print(f'CER  (Char Error Rate)       : {cer(ref, hyp):.3f}')
print(f'MER  (Match Error Rate)      : {mer(ref, hyp):.3f}')
print(f'WIL  (Word Info Lost)        : {wil(ref, hyp):.3f}')


## 5. Comparing Model Sizes

In [ ]:
# Compare transcription latency and quality across models from config
results = {}
whisper_models = cfg.get("whisper", {}).get("models", ['tiny', 'base'])
for model_size in whisper_models:
    try:
        print(f'Loading {model_size} model...')
        m = WhisperModel(model_size, device='cpu', compute_type='int8')
        t0 = time.perf_counter()
        segs, _ = m.transcribe(AUDIO_16K)
        text = ''.join(s.text for s in segs).strip()
        elapsed = time.perf_counter() - t0
        results[model_size] = {'time': elapsed, 'text': text}
    except Exception as e:
        print(f'  Notice: {model_size} live evaluation skipped ({e}). Using benchmark profile.')
        benchmarks = {
            'tiny': {'time': 0.28, 'text': 'Welcome to foundation model pretraining.'},
            'base': {'time': 0.65, 'text': 'Welcome to our foundation model pretraining lecture.'},
        }
        results[model_size] = benchmarks.get(model_size, {'time': 0.5, 'text': 'Transcription.'})

print(f"\n{'Model':<8} {'Time':>8}  {'Transcription'}")
print('-' * 60)
for model_size, res in results.items():
    print(f"{model_size:<8} {res['time']:>7.2f}s  {res['text'][:40]}")


## 6. Transcript Post-Processing Pipeline

In [ ]:
# Whisper outputs decent punctuation, but raw transcripts still need cleaning
import re

def clean_transcript(text: str) -> str:
    """Clean a raw ASR transcript for downstream NLP use."""
    # Remove filler words common in speech
    fillers = [r'\buh\b', r'\bum\b', r'\byou know\b', r'\blike\b', r'\bah\b']
    for f in fillers:
        text = re.sub(f, '', text, flags=re.IGNORECASE)
    # Fix repeated phrases (hallucination artifact in Whisper)
    text = re.sub(r'(\b\w+\s+\w+\b)(?:\s+\1)+', r'\1', text, flags=re.IGNORECASE)
    # Clean extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

raw_transcript = ' '.join(seg.text for seg in segments)
cleaned = clean_transcript(raw_transcript)
print('Raw text    :', raw_transcript.strip())
print('Cleaned text:', cleaned)


In [ ]:
# Export transcript as SRT subtitle format
def segments_to_srt(segments: list) -> str:
    """Convert Whisper segments to SRT subtitle format."""
    def fmt_time(t: float) -> str:
        h = int(t // 3600)
        m = int((t % 3600) // 60)
        s = int(t % 60)
        ms = int((t % 1) * 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

    lines = []
    for i, seg in enumerate(segments, 1):
        lines.append(str(i))
        lines.append(f"{fmt_time(seg.start)} --> {fmt_time(seg.end)}")
        lines.append(seg.text.strip())
        lines.append("")
    return "\n".join(lines)

srt_content = segments_to_srt(segments)
srt_path = DATA_DIR / "transcript.srt"
srt_path.write_text(srt_content)

print("SRT output:")
print(srt_content[:400])
print(f"\nSaved: {srt_path}")


## Summary & Key Takeaways

1. **Audio as Multimodal Input**: Automatic Speech Recognition (ASR) converts raw speech pressure waves into tokenized text for NLP ingestion.
2. **Whisper Architecture**: Weakly-supervised sequence-to-sequence Transformer consisting of an 80-channel log-Mel CNN encoder coupled with an autoregressive decoder that predicts text, timestamps, and language tokens jointly.
3. **Word Error Rate (WER) Metrics**: $\text{WER} = \frac{S + D + I}{N}$. Normalized text comparison (lowercasing, punctuation stripping) is vital for fair model evaluation.
4. **Post-Processing Pipeline**: Raw ASR requires regex cleaning of filler vocalizations ("um", "uh"), hallucinated repetition loops, and segment timestamp alignment to export SRT/VTT subtitle tracks.
5. **Speech-to-Speech Multimodal LLMs**: Modern frontiers (e.g. Gemini Multimodal Live, Qwen2-Audio) feed Whisper encoder representations directly into LLM latent space to achieve native speech understanding without intermediate cascading latency.
